In [1]:
import os
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import train_test_split

# Paths
kalman_filter_data_path = '../data/kaggle-drdataboston/attempt_2/updated_data_kalman_filtered'
matrix_path = '../data/kaggle-drdataboston/matrix.csv'
out_path = '../data/kaggle-drdataboston/attempt_2/regression_split'

# Parameters
window_ms = 5000
interp_pts = 500

# Load metadata
matrix = pd.read_csv(matrix_path)

# Step 1: Split matrix.csv into train/valid/test
person_ids = matrix.iloc[:, 0].values  # Assuming first column is the ID

train_ids, test_ids = train_test_split(person_ids, test_size=0.2, random_state=42)
train_ids, val_ids = train_test_split(train_ids, test_size=0.25, random_state=42)  # 0.25 x 0.8 = 0.2

splits = {
    "train": set(train_ids),
    "valid": set(val_ids),
    "test": set(test_ids)
}

# Initialize storage
data = {
    "train": {"X": [], "age": [], "height": [], "weight": [], "gender": []},
    "valid": {"X": [], "age": [], "height": [], "weight": [], "gender": []},
    "test": {"X": [], "age": [], "height": [], "weight": [], "gender": []},
}

start_time = time.perf_counter()

for idx, file in enumerate(os.listdir(kalman_filter_data_path)):
    if idx % 5 == 0:
        c_time = time.perf_counter()
        print(f"Processed {idx} files in {c_time - start_time:.6f} seconds")

    if not file.endswith('.csv'):
        continue

    file_path = os.path.join(kalman_filter_data_path, file)
    df = pd.read_csv(file_path)

    clean_name = file
    if clean_name.endswith('.csv.csv'):
        clean_name = clean_name[:-4]

    person_id = None
    for i, row in matrix.iterrows():
        if clean_name.strip().lower() in [str(row.iloc[j]).strip().lower() for j in range(1, 5)]:
            person_id = row.iloc[0]
            weight = row.iloc[5]
            age = row.iloc[6]
            height = row.iloc[7]
            gender = row.iloc[11]
            break

    if person_id is None:
        continue

    # Assign to correct split
    split = None
    for k in splits:
        if person_id in splits[k]:
            split = k
            break

    if split is None:
        continue

    df['window_id'] = (df['timestamp'] - df['timestamp'].iloc[0]) // window_ms

    for _, group in df.groupby('window_id'):
        if len(group) < 2:
            continue

        time_arr = group['timestamp'].values
        mag = group['filtered'].values
        time_arr = time_arr - time_arr[0]

        new_time = np.linspace(0, time_arr[-1], interp_pts)
        interp_mag = np.interp(new_time, time_arr, mag)

        data[split]["X"].append(interp_mag)
        data[split]["age"].append(age)
        data[split]["height"].append(height)
        data[split]["weight"].append(weight)
        data[split]["gender"].append(gender)

# Save each set
os.makedirs(out_path, exist_ok=True)
for split in ["train", "valid", "test"]:
    np.save(f"{out_path}/X_{split}.npy", np.array(data[split]["X"]))
    np.save(f"{out_path}/y_age_{split}.npy", np.array(data[split]["age"]))
    np.save(f"{out_path}/y_height_{split}.npy", np.array(data[split]["height"]))
    np.save(f"{out_path}/y_weight_{split}.npy", np.array(data[split]["weight"]))
    np.save(f"{out_path}/y_gender_{split}.npy", np.array(data[split]["gender"]))


Processed 0 files in 0.000921 seconds
Processed 5 files in 0.240349 seconds
Processed 10 files in 0.448876 seconds
Processed 15 files in 0.656392 seconds
Processed 20 files in 0.863033 seconds
Processed 25 files in 0.984361 seconds
Processed 30 files in 1.159006 seconds
Processed 35 files in 1.609066 seconds
Processed 40 files in 1.993539 seconds
Processed 45 files in 2.286828 seconds
Processed 50 files in 2.636864 seconds
Processed 55 files in 2.762411 seconds
Processed 60 files in 2.872888 seconds
Processed 65 files in 2.980251 seconds
Processed 70 files in 3.088116 seconds
Processed 75 files in 3.404922 seconds
Processed 80 files in 3.573485 seconds
Processed 85 files in 3.758201 seconds
Processed 90 files in 3.878244 seconds
Processed 95 files in 3.999307 seconds
Processed 100 files in 4.129220 seconds
Processed 105 files in 4.427098 seconds
Processed 110 files in 4.720697 seconds
Processed 115 files in 4.858297 seconds
Processed 120 files in 5.064910 seconds
Processed 125 files in

In [3]:
# Check loaded data shapes and sample values
for split in ["train", "valid", "test"]:
    X = np.load(f"{out_path}/X_{split}.npy")
    y_age = np.load(f"{out_path}/y_age_{split}.npy")
    y_height = np.load(f"{out_path}/y_height_{split}.npy")
    y_weight = np.load(f"{out_path}/y_weight_{split}.npy")
    y_gender = np.load(f"{out_path}/y_gender_{split}.npy")

    print(f"=== {split.upper()} ===")
    print("X shape:", X.shape)
    print("Age shape:", y_age.shape)
    print("Height shape:", y_height.shape)
    print("Weight shape:", y_weight.shape)
    print("Gender shape:", y_gender.shape)
    
    print("Sample gender values:", y_gender[:5])
    print("Sample height values:", y_height[:5])
    print("Sample weight values:", y_weight[:5])
    print("Sample age values:   ", y_age[:5])
    print("-" * 40)


=== TRAIN ===
X shape: (15739, 500)
Age shape: (15739,)
Height shape: (15739,)
Weight shape: (15739,)
Gender shape: (15739,)
Sample gender values: ['F' 'F' 'F' 'F' 'F']
Sample height values: [165 165 165 165 165]
Sample weight values: [44 44 44 44 44]
Sample age values:    [19 19 19 19 19]
----------------------------------------
=== VALID ===
X shape: (5917, 500)
Age shape: (5917,)
Height shape: (5917,)
Weight shape: (5917,)
Gender shape: (5917,)
Sample gender values: ['M' 'M' 'M' 'M' 'M']
Sample height values: [182 182 182 182 182]
Sample weight values: [86 86 86 86 86]
Sample age values:    [47 47 47 47 47]
----------------------------------------
=== TEST ===
X shape: (4432, 500)
Age shape: (4432,)
Height shape: (4432,)
Weight shape: (4432,)
Gender shape: (4432,)
Sample gender values: ['M' 'M' 'M' 'M' 'M']
Sample height values: [175 175 175 175 175]
Sample weight values: [52 52 52 52 52]
Sample age values:    [18 18 18 18 18]
----------------------------------------
